In [1]:
import os
os.sys.path.append('/data/scratch/globc/bonassies/workspace/swot_for_flood')

import configparser
from pathlib import Path
from core.swot_project import SwotProject
from core.swot_raster import SwotRaster

main_path = "/data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Mozambique"

# Download and rasterize notebook

This notebook explain how to use the swot_for_flood library to download and rasterize the SWOT data for a given region and time period. 

## Define a project

This library is designed to work with a project. A project is a directory that contains the configuration file `config.cfg` and the data. The configuration file contains the parameters of the project.

In this notebook, we will create a project named "example_download_rasterize" in the directory "examples". The project will contain the SWOT data for the region of interest and time period defined in the configuration file.

In [2]:
config = configparser.ConfigParser()
config.read(main_path + '/config.cfg')

print(type(config),dict(config['CONFIG']))

<class 'configparser.ConfigParser'> {'project': 'Mozambique', 'workspace': '/data/scratch/globc/bonassies/workspace/swot_for_flood/examples', 'data_path': '/data/scratch/globc/bonassies/data', 'aoi': '/data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Mozambique/aux_data/AOI_Mozambique_32736.gpkg', 'floodmask_path': '/data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Mozambique/aux_data/AOI_Mozambique_32736.gpkg', 'controlmask_path': '/data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Mozambique/aux_data/AOI_Mozambique_32736.gpkg', 'esa_worldcover_path': '/data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Mozambique/aux_data/ESA_WC_10m_cut_AOI_nrow1502_ncol2002.tif', 'crs': '32736', 'aoi_crs': '32736', 'first_time': '2025-01-01', 'last_time': '2026-02-01', 'nodes': '8', 'download_type': 'PIXC', 'list_dry_dates': '[2025-05-24, 2025-06-14, 2025-07-05, 2025-07-26, 2025-08-16, 2025-09-06, 2025-09-26, 2025-10-17, 2025-11-07, 2025-11-28, 20

The config can also be a dictionary

In [3]:
# from pathlib import Path

# param_dict = {
#     'project': 'Greece_EMSR692',
#     'workspace': Path("/data/scratch/globc/bonassies/workspace"),
#     'data_path': Path("/data/scratch/globc/bonassies/data"),
#     'CRS': 'EPSG:32634',
#     'first_time': "2023-09-05",
#     'last_time': "2023-09-20",
#     'aoi': Path("/data/scratch/globc/bonassies/workspace").joinpath('Greece_EMSR692',"aux_data","EMSR692_aois_V2.gpkg"),
#     'aoi_crs': 'EPSG:4326',
#     'passes': [402, 417],
#     'nodes': 8,
#     'do_download': False,
#     'download_type': 'PIXC',
#     'GDAL_NUM_THREADS': 10,
#     'GDAL_CACHEMAX': 160000,
#     'do_make_gpkg': False,
#     'do_make_tiff': False    
# }

Once the config file loaded, we can use it to create a project object. The project object will contain the parameters of the project and the data.

In [3]:
# config["CONFIG"]['do_make_gpkg'] = 'True'
# config["CONFIG"]['do_make_tiff'] = 'True'
swot_project = SwotProject(config)

# print(swot_project)


Data path already exists in /data/scratch/globc/bonassies/data or download is set to False
SWOT data already exists in /data/scratch/globc/bonassies/data/SWOT or download is set to False
SWOT project already exists in /data/scratch/globc/bonassies/data/SWOT/Mozambique or download is set to False
Geopackage already exists in /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Mozambique/gpkg_combined or make_gpkg is set to False
TIFF path already exists in /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Mozambique/rasters or make_tiff is set to False
No automatic download, please use the Downloader object to download the data


On the first time, you should get an error because the data is not downloaded yet. The project object will download the data and store it in the project directory only if do_download is set to True in the configuration file.

You can manually download the data by calling the download method of the project object.

First we need to search for the data we want to download. We can use the search method of the project object to search for the data. The search method will return a list of the data that match the search criteria.

In [ ]:
swot_project.Downloader.search_PIXC(True)
# swot_project.Downloader.search_PIXCVec(True)
# swot_project.Downloader.search_Nodes(True)
# swot_project.Downloader.search_Reachs(True)

Then we can download the data by calling the download method of the project object. The download method will download the data and store it in the project directory.

In [ ]:
swot_project.Downloader.download_pool() # if multithreading download
# swot_project.Downloader.download_granules() # if single thread download


Finally we can rasterize the data by calling the rasterize method of the project object. The rasterize method will rasterize the data and store it in the project directory.

First we create gpgk file that combine the netcdf files and then we rasterize the data.

In [5]:
swot_project.Rasterizer.SWOT_PATH.name
list_tile = [name.name for name in swot_project.Rasterizer.SWOT_PATH.glob(f'*PIXC*.nc')]
tiles = [name.split('_')[5] + "_" + name.split('_')[6] for name in list_tile]
tiles = list(set(tiles))
passes = list(set([name.split('_')[0] for name in tiles]))
tiles

['165_111R']

In [6]:
swot_project.Rasterizer.tile_names_selection = []
for pass_num in passes:
    sublist_tile = [name for name in tiles if pass_num in name]
    swot_project.Rasterizer.tile_names_selection.append(sublist_tile)
    
swot_project.Rasterizer.tile_names_selection

[['165_111R']]

In [8]:
swot_project.Rasterizer.make_space = True
swot_project.Rasterizer.find_pixc() #True : only pre-treat the ganules on the studied dates (based on dry and flooded dates lists)
# swot_project.Rasterizer.pixc_to_gpkg() #True : only pre-treat the ganules on the studied dates (based on dry and flooded dates lists)

In [ ]:
swot_project.Rasterizer.gpkg_to_tiff(
    power=2,
    smoothing=1,
    radius= 20,
    max_points=20, 
    )

>>> Converting the SWOT geopackage to tiff
>>> Generate tiff for every variables
Working on  /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Mozambique/gpkg_combined/SWOT_epsg32736_20260130T033259.gpkg
>>> Generate tiff for  sig0
gdal_grid -a invdistnn:power=2:smoothing=1:radius=20:max_points=3:nodata=-9999 -txe 551269.0312166283 571283.8956532273 -tye 7239838.132997731 7224825.695491079 -outsize 2002.0 1502.0 -zfield sig0 -of GTiff -ot Float32 /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Mozambique/gpkg_combined/SWOT_epsg32736_20260130T033259.gpkg /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Mozambique/rasters/sig0/SWOT_epsg32736_20260130T033259_sig0.tif --config GDAL_NUM_THREADS 11 --config GDAL_CACHEMAX 160000
Grid data type is "Float32"
Grid size = (2002 1502).
Corner coordinates = (551269.031217 7239838.132998)-(571283.895653 7224825.695491).
Grid cell size = (9.997435 -9.994965).
Source point count = 1099636.
Algorithm name: "

ERROR 6: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Mozambique/rasters/SWOT_epsg32736_20250524T183203_combined.tif, band 1: SetColorTable() not supported for multi-sample TIFF files.



Processing file     2 of     5, 20.000% completed in 0 minutes.
Filename: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Mozambique/rasters/coherent_power/SWOT_epsg32736_20250524T183203_coherent_power.tif
File Size: 2002x1502x1
Pixel Size: 9.997435 x -9.994965
UL:(551269.031217,7239838.132998)   LR:(571283.895653,7224825.695491)
Copy 0,0,2002,1502 to 0,0,2002,1502.

Processing file     3 of     5, 40.000% completed in 0 minutes.
Filename: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Mozambique/rasters/gamma_tot/SWOT_epsg32736_20250524T183203_gamma_tot.tif
File Size: 2002x1502x1
Pixel Size: 9.997435 x -9.994965
UL:(551269.031217,7239838.132998)   LR:(571283.895653,7224825.695491)
Copy 0,0,2002,1502 to 0,0,2002,1502.

Processing file     4 of     5, 60.000% completed in 0 minutes.
Filename: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Mozambique/rasters/gamma_SNR/SWOT_epsg32736_20250524T183203_gamma_SNR.tif
File Size: 2002x1502x1
Pi

ERROR 6: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Mozambique/rasters/SWOT_epsg32736_20250614T151707_combined.tif, band 1: SetColorTable() not supported for multi-sample TIFF files.



Processing file     2 of     5, 20.000% completed in 0 minutes.
Filename: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Mozambique/rasters/coherent_power/SWOT_epsg32736_20250614T151707_coherent_power.tif
File Size: 2002x1502x1
Pixel Size: 9.997435 x -9.994965
UL:(551269.031217,7239838.132998)   LR:(571283.895653,7224825.695491)
Copy 0,0,2002,1502 to 0,0,2002,1502.

Processing file     3 of     5, 40.000% completed in 0 minutes.
Filename: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Mozambique/rasters/gamma_tot/SWOT_epsg32736_20250614T151707_gamma_tot.tif
File Size: 2002x1502x1
Pixel Size: 9.997435 x -9.994965
UL:(551269.031217,7239838.132998)   LR:(571283.895653,7224825.695491)
Copy 0,0,2002,1502 to 0,0,2002,1502.

Processing file     4 of     5, 60.000% completed in 0 minutes.
Filename: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Mozambique/rasters/gamma_SNR/SWOT_epsg32736_20250614T151707_gamma_SNR.tif
File Size: 2002x1502x1
Pi

...


Processing file     2 of     5, 20.000% completed in 0 minutes.
Filename: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Mozambique/rasters/coherent_power/SWOT_epsg32736_20251219T100249_coherent_power.tif
File Size: 2002x1502x1
Pixel Size: 9.997435 x -9.994965
UL:(551269.031217,7239838.132998)   LR:(571283.895653,7224825.695491)
Copy 0,0,2002,1502 to 0,0,2002,1502.

Processing file     3 of     5, 40.000% completed in 0 minutes.
Filename: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Mozambique/rasters/gamma_tot/SWOT_epsg32736_20251219T100249_gamma_tot.tif
File Size: 2002x1502x1
Pixel Size: 9.997435 x -9.994965
UL:(551269.031217,7239838.132998)   LR:(571283.895653,7224825.695491)
Copy 0,0,2002,1502 to 0,0,2002,1502.

Processing file     4 of     5, 60.000% completed in 0 minutes.
Filename: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Mozambique/rasters/gamma_SNR/SWOT_epsg32736_20251219T100249_gamma_SNR.tif
File Size: 2002x1502x1
Pi

ERROR 6: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Mozambique/rasters/SWOT_epsg32736_20260109T064754_combined.tif, band 1: SetColorTable() not supported for multi-sample TIFF files.



Processing file     2 of     5, 20.000% completed in 0 minutes.
Filename: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Mozambique/rasters/coherent_power/SWOT_epsg32736_20260109T064754_coherent_power.tif
File Size: 2002x1502x1
Pixel Size: 9.997435 x -9.994965
UL:(551269.031217,7239838.132998)   LR:(571283.895653,7224825.695491)
Copy 0,0,2002,1502 to 0,0,2002,1502.

Processing file     3 of     5, 40.000% completed in 0 minutes.
Filename: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Mozambique/rasters/gamma_tot/SWOT_epsg32736_20260109T064754_gamma_tot.tif
File Size: 2002x1502x1
Pixel Size: 9.997435 x -9.994965
UL:(551269.031217,7239838.132998)   LR:(571283.895653,7224825.695491)
Copy 0,0,2002,1502 to 0,0,2002,1502.

Processing file     4 of     5, 60.000% completed in 0 minutes.
Filename: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Mozambique/rasters/gamma_SNR/SWOT_epsg32736_20260109T064754_gamma_SNR.tif
File Size: 2002x1502x1
Pi

ERROR 6: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Mozambique/rasters/SWOT_epsg32736_20260130T033259_combined.tif, band 1: SetColorTable() not supported for multi-sample TIFF files.



Processing file     2 of     5, 20.000% completed in 0 minutes.
Filename: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Mozambique/rasters/coherent_power/SWOT_epsg32736_20260130T033259_coherent_power.tif
File Size: 2002x1502x1
Pixel Size: 9.997435 x -9.994965
UL:(551269.031217,7239838.132998)   LR:(571283.895653,7224825.695491)
Copy 0,0,2002,1502 to 0,0,2002,1502.

Processing file     3 of     5, 40.000% completed in 0 minutes.
Filename: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Mozambique/rasters/gamma_tot/SWOT_epsg32736_20260130T033259_gamma_tot.tif
File Size: 2002x1502x1
Pixel Size: 9.997435 x -9.994965
UL:(551269.031217,7239838.132998)   LR:(571283.895653,7224825.695491)
Copy 0,0,2002,1502 to 0,0,2002,1502.

Processing file     4 of     5, 60.000% completed in 0 minutes.
Filename: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Mozambique/rasters/gamma_SNR/SWOT_epsg32736_20260130T033259_gamma_SNR.tif
File Size: 2002x1502x1
Pi

If needed, we can put extra rasters to the same resolution and extent as the SWOT data. This is useful to compare the SWOT data with other data.

It uses gdal to rasterize the data. gdal is used as command line using os.system.

In [5]:
raster = Path("/data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Mozambique/aux_data/ESA_WC_ANIM_32736.tif")
raster_crs = 32736
interp = 'nearest'
swot_project.Rasterizer.gdalwarp_raster_to_swot_bbox_and_size(raster, raster_crs, interp)


>>> Converting the AUXILARY rasters to tiff
gdalwarp -s_srs EPSG:32736 -t_srs EPSG:32736 -te 554842.3354214092 7235983.940609195 568995.5802669114 7245484.360314356 -ts 1416.0 951.0 -r nearest -of GTiff /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Mozambique/aux_data/ESA_WC_ANIM_32736.tif /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Mozambique/aux_data/ESA_WC_ANIM_32736_nrow951_ncol1416.tif
Copying color table from /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Mozambique/aux_data/ESA_WC_ANIM_32736.tif to new file.
Creating output file that is 1416P x 951L.
Processing /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Mozambique/aux_data/ESA_WC_ANIM_32736.tif [1/1] : 0Using internal nodata values (e.g. 0) for image /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Mozambique/aux_data/ESA_WC_ANIM_32736.tif.
Copying nodata values from source /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Mo

You can check other notebooks for more information about the library.